In [0]:

import json

source_data = [
    {
        "customer_id": 101,
        "customer_name": "Anu",
        "address": {
            "city": "Mumbai",
            "state": "Maharashtra"
        },
        "orders": [
            {
                "order_id": 5001,
                "order_date": "2026-10-10",
                "items": [
                    {"product_id": "P101", "quantity": 2, "amount": 500.0},
                    {"product_id": "P102", "quantity": 1, "amount": 300.0}
                ]
            },
            {
                "order_id": 5002,
                "order_date": "2026-10-11",
                "items": [
                    {"product_id": "P103", "quantity": 3, "amount": 900.0}
                ]
            }
        ]
    },
    {
        "customer_id": 102,
        "customer_name": "Rahul",
        "address": {
            "city": "Pune",
            "state": "Maharashtra"
        },
        "orders": [
            {
                "order_id": 5003,
                "order_date": "2026-10-12",
                "items": [
                    {"product_id": "P104", "quantity": 1, "amount": 250.0}
                ]
            }
        ]
    }
]

json_string = json.dumps(source_data, indent=2)

print(json_string)


file_path = "/Volumes/reconciliation/default/reconciliation_files/retail_source.json"

with open(file_path, "w") as file:
    file.write(json_string)

print("JSON file saved successfully!")
print(file_path)



In [0]:

file_path = "/Volumes/reconciliation/default/reconciliation_files/retail_source.json"

source_df = spark.read.option("multiLine", True).json(file_path)

display(source_df)


In [0]:
source_df.printSchema()

In [0]:

from pyspark.sql.functions import col, explode

orders_df = source_df.select(
    col("customer_id"),
    col("customer_name"),
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    explode(col("orders")).alias("order")
)

display(orders_df)


In [0]:


items_df = orders_df.select(
    col("customer_id"),
    col("customer_name"),
    col("city"),
    col("state"),
    col("order.order_id").alias("order_id"),
    col("order.order_date").alias("order_date"),
    explode(col("order.items")).alias("item")
)

display(items_df)



In [0]:


final_df = items_df.select(
    col("customer_id"),
    col("customer_name"),
    col("city"),
    col("state"),
    col("order_id"),
    col("order_date"),
    col("item.product_id").alias("product_id"),
    col("item.quantity").alias("quantity"),
    col("item.amount").alias("amount")
)

display(final_df)


In [0]:

table_name = "reconciliation.default.retail_orders"

final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print("Data successfully loaded into Delta table:", table_name)


In [0]:

target_df = spark.table("reconciliation.default.retail_orders")

display(target_df)

print("Target record count:", target_df.count())
